<a href="https://colab.research.google.com/github/jlra5/TFG-Jose-Luis-Rodriguez-Aparicio/blob/main/S4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalar Ollama

!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tgz
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
import subprocess
import time

def start_ollama_server():
    subprocess.Popen(['ollama', 'serve'])
    time.sleep(5)  # Espera a que se inicie
    print("✅ Ollama server iniciado en http://localhost:11434")

start_ollama_server()

✅ Ollama server iniciado en http://localhost:11434


In [3]:
# Descargar qwen2.7:7B

import subprocess

result = subprocess.run(['ollama', 'pull', 'qwen2.5:7b'],
                       capture_output=True, text=True)
print(result.stdout)


In [4]:
# Check model is installed

!ollama list

NAME          ID              SIZE      MODIFIED      
qwen2.5:7b    845dbda0ea48    4.7 GB    5 seconds ago    


In [5]:
# GPU

!nvidia-smi --query-gpu=name,memory.total --format=csv,nounits

name, memory.total [MiB]
Tesla T4, 15360


In [6]:
# Install dependencies

import subprocess, sys
pkgs = ["gradio", "langgraph", "langchain", "langchain-core",
        "langchain-community", "langchain-ollama", "pydantic", "requests", "time", "grandalf", "random"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p])
print("✅ Dependencias instaladas")

✅ Dependencias instaladas


In [16]:
from __future__ import annotations
#-----------------------------------------------------------------------------------------------------------------------
# Block 0 - Imports
#-----------------------------------------------------------------------------------------------------------------------
import json
import time
import gradio as gr
import random
import logging
import warnings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AnyMessage, BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langchain.tools import tool
from langgraph.graph.message import add_messages
from operator import add
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated, Literal, Optional, Any

CONFIDENCE_THRESHOLD = 0.8

LOG_NAME = "logfile.log"

logger = logging.getLogger(LOG_NAME)
logger.setLevel(logging.DEBUG)
logger.propagate = False

handler = logging.FileHandler(LOG_NAME)
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

warnings.filterwarnings("ignore", category=UserWarning)
#-----------------------------------------------------------------------------------------------------------------------
# Block 2 - State Definition
#-----------------------------------------------------------------------------------------------------------------------

class State(TypedDict):
    '''
    Shared state amongst graph nodes.
    '''
    messages: Annotated[list, add_messages]     # System messages

    instruction: Optional[str]
    order_id: Optional[int]
    product_id: Optional[int]
    customer_group: Optional[int]

    intention: Optional[str]                    # Identified instruction
    chosen_tool: Optional[str]                  # Matching tool to instruction
    tool_confidence: Optional[float]            # % Confidence of the tool
    tool_params: Optional[dict]                 # Parameters of the tool
    params_schema: Optional[str]
    total_tokens: Annotated[int, add]
    error_descr: Optional[str]
    tool_definition: Optional[str]


#-----------------------------------------------------------------------------------------------------------------------
# Block 3 - Tool definition
#-----------------------------------------------------------------------------------------------------------------------

# Tool Schemas for each tool

class ParamsDelayOrder(BaseModel):
    reasoning: str = Field(description="Briefly explain the extracted parameters"),
    order_id: int = Field(description="id of the order to be updated")
    new_date: str = Field(description="new updated delivery date")

class ParamsCancelOrderLine(BaseModel):
    reasoning: str = Field(description="Briefly explain the extracted parameters"),
    order_id: int = Field(description="id of the order to be updated")
    product_id: int = Field(description="id of the product whose quantity is to be set to zero")

class ParamsSwapProducts(BaseModel):
    reasoning: str = Field(description="Briefly explain the extracted parameters"),
    order_id: int = Field(description="id of the order to be updated")
    old_product_id: int = Field(description="id of the old product to be replaced")
    new_product_id: int = Field(description="id of the new product to be inserted")

class ParamsChangeProductQuantity(BaseModel):
    reasoning: str = Field(description="Briefly explain the extracted parameters"),
    order_id: int = Field(description="id of the order to be updated")
    product_id: int = Field(description="id of the product whose quantity is to be changed")
    new_quantity: int = Field(description="new quantity to be changed")

class ParamsUnblockCredit(BaseModel):
    reasoning: str = Field(description="Briefly explain the extracted parameters"),
    order_id: int = Field(description="id of the order to be updated")

class ParamsChangeUnknown(BaseModel):
    pass

TOOL_SCHEMAS = {
    "delay_order": ParamsDelayOrder,
    "cancel_order_line": ParamsCancelOrderLine,
    "swap_products": ParamsSwapProducts,
    "change_product_quantity": ParamsChangeProductQuantity,
    "unblock_credit": ParamsUnblockCredit,
    "unknown": ParamsChangeUnknown
}

# Define tools

@tool
def delay_order(
        order_id: int = Field(description="id of the order to be updated. It is a 9 digit number. Example:123456789"),
        new_date: str= Field(description="new updated delivery date. It is coded as dd/mm/yy. Example: 15/01/26")
) -> dict:
    """
    Changes the delivery date of the order with order_id to the date new_date.
    Use it when the intention is to delay the order.
    """
    action = f"delay_order(order_id={order_id}, new_date={new_date})"
    number = random.random() #Simulating tool execution error
    if number > 0.8:
        error_descr = f"Error - {action}"
        logger.info(f"TOOL EXECUTION - Error: {error_descr}")
    else:
        error_descr = ""
        logger.info(f"TOOL EXECUTION - Executing {action}")

    return {
        "error_descr": error_descr,
        "chosen_tool": action
    }

@tool
def cancel_order_line(
        order_id: int = Field(description="id of the order to be updated. It is a 9 digit number. Example:123456789"),
        product_id: int = Field(description="""id of the product whose quantity is to be set to zero. Is is a 6 digit
                                            number. Example:123456""")
) -> str:
    """
    Cancels the order line belonging to the product. This means to set the product quantity to zero.
    Use it when the intention is to cancel the order.
    """
    action = f"cancel_order_line(order_id={order_id}, product_id={product_id})"
    number = random.random() #Simulating tool execution error
    if number > 0.8:
        error_descr = f"Error - {action}"
        logger.info(f"TOOL EXECUTION - Error: {error_descr}")
    else:
        error_descr = ""
        logger.info(f"TOOL EXECUTION - Executing {action}")

    return {
        "error_descr": error_descr,
        "chosen_tool": action
    }

@tool
def swap_products(
        order_id: int = Field(description="id of the order to be updated. It is a 9 digit number. Example:123456789"),
        old_product_id: int = Field(description="id of the old product to be replaced. It is a 6 digit number. Example:123456") ,
        new_product_id: int = Field(description="id of the new product to be inserted. It is a 6 digit number. Example:123456")
) -> str:
    """
    Swaps old product with the new product in the order.
    Use it when the intention is to swap products in an order
    """
    action = f"swap_products(order_id={order_id}, old_product_id={old_product_id}, new_product_id={new_product_id})"
    number = random.random() #Simulating tool execution error
    if number > 0.8:
        error_descr = f"Error - {action}"
        logger.info(f"TOOL EXECUTION - Error: {error_descr}")
    else:
        error_descr = ""
        logger.info(f"TOOL EXECUTION - Executing {action}")

    return {
        "error_descr": error_descr,
        "chosen_tool": action
    }


@tool
def change_product_quantity(
        order_id: int = Field(description="id of the order to be updated. It is a 9 digit number. Example:123456789"),
        product_id: int = Field(description="id of the product whose quantity is to be changed. It is a 6 digit number. Example:123456"),
        new_quantity: int = Field(description="Quantity to be inserted. It is a number. Example: 3000 units")
)-> str:
    """
    Updates the product quantity of the order line belonging to the product.
    Use it when the intention is to change the product quantity.
    """
    action = f"change_product_quantity(order_id={order_id}, product_id={product_id}, new_quantity={new_quantity})"
    number = random.random() #Simulating tool execution error
    if number > 0.8:
        error_descr = f"Error - {action}"
        logger.info(f"TOOL EXECUTION - Error: {error_descr}")
    else:
        error_descr = ""
        logger.info(f"TOOL EXECUTION - Executing {action}")

    return {
        "error_descr": error_descr,
        "chosen_tool": action
    }

@tool
def unblock_credit(
    order_id: Optional[int] = Field(
        description=("Order ID whose credit block must be released. ")
    )
) -> str:
    """
    Release an order that is blocked due to credit limit issues.
    Use this tool when the instruction requests to liberate, release,
    or remove a credit block on the order.
    """
    action = f"unblock_credit(order_id={order_id})"
    number = random.random() #Simulating tool execution error
    if number > 0.8:
        error_descr = f"Error - {action}"
        logger.info(f"TOOL EXECUTION - {error_descr}")
    else:
        error_descr = ""
        logger.info(f"TOOL EXECUTION - Executing {action}")

    return {
        "error_descr": error_descr,
        "chosen_tool": action
    }

tools = {
    "delay_order": delay_order,
    "cancel_order_line": cancel_order_line,
    "swap_products": swap_products,
    "change_product_quantity": change_product_quantity,
    "unblock_credit": unblock_credit
}


#-----------------------------------------------------------------------------------------------------------------------
# Block 3 - Instantiate the model and bind tools
#-----------------------------------------------------------------------------------------------------------------------

options = {
    'temperature': 0,
    'top_k': 1,
    'top_p': 0.1,
    'repeat_penalty': 1.2
}

logger.info("MAIN - Setting up model")
llm = ChatOllama(
    #model="mistral",
    model="qwen2.5:7b",
    #model="qwen2.5-optimized",
    #model="qwen3:8B",
    options=options
)

# Schema to force the output of the call to the SLM/LLM
# Reasoning to improve order bias
class Intention(BaseModel):
    """ Schema to restrict/focus SLM/LLM answers"""
    reasoning: str = Field(description="Briefly explain the extracted customer group and availability status derived from the instruction."),
    intention: Literal["delay_order", "cancel_order_line", "swap_products", "change_product_quantity", "unblock_credit", "unknown"]
    tool_confidence: float

#-----------------------------------------------------------------------------------------------------------------------
# Block 5 - Node definition
#-----------------------------------------------------------------------------------------------------------------------

def node_read_instruction(state: State) -> dict:
    """
    Node to read the instructions and set their intention.
    """

    # Reads the instruction field of the message
    # Uses the model (SLM/Embeddings/other) to find the most suitable tool and its confidence
    # structured_output could be used to limit model's variability
    # If no tool is found, return "Unknown"

    logger.info(f"MODEL NODE - Starting")
    error_descr = ""
    messages = state["messages"]
    if not messages:
        looger.info(f"MODEL NODE - No messages")
        logger.info(f"MODEL NODE - Quitting node")
        return {
            "messages": [AIMessage(content=f"Intention: Unknown")],
            "instruction": "Unknown",
            "error_descr": error_descr,}

    instruction = state["instruction"]
    customer_group = state["customer_group"]

    # We force the output to be one option of a predefined list
    llm_with_structure = llm.with_structured_output(Intention, include_raw=True)

    # Prompt for intention assert
    prompt = ChatPromptTemplate.from_messages([
        ("system", """
        You are ane expert in machine learning. Your task is to capture the intention of the user.

        ### INPUTS:
        Customer group: {customer_group}
        Instruction: {instruction}

        ### POSSIBLE INTENTIONS:
        - "cancel_order_line": if customer group is 2 and the instruction shows no availability or later availability date.
        - "delay_order": if customer group is 1 and the instruction shows no availability or later availability date.
        - "swap_products": to be used when the instruction indicates a different product id.
        - "change_product_quantity": to be used when the instruction indicates a different product quantity.
        - "unblock_credit": to be used when the instruction refers to releasing an order or a credit block.
        - "unknown": to be used when the instruction is irrelevant, nonsense or lacks specific logistic instructions like product, dates or stock status.

        ### CLASSIFICACTION RULES:
        - If product is out of stock or the availability is at a later date or requested to delay an order, then:
            * If customer group is 1, then return "delay_order"
            * If customer group is 2, then return "cancel_order_line"

        - If the instruction mentions a different product id, then return "swap_products"
        - If the instruction mentions a different quantity, then return "change_product_quantity"
        - If the instruction mentions a release or order liberation, the return "unblock_credit"
        - If the intents are not aligned with the instruction, then return "Unknown".
        - If the customer group is 2, the intent is never delay_order, use cancel_order_line instead.


        Do not assume that the product is out of stock or the availability is at a later date.

        Dates are expressed in dd/mm/yy format. Quantities are integers. Product id are integers of six digits.

        ### EXAMPLES
        - "Product avaiable on 15/01/26, customer group 1": delay the order
        - "Product avaiable on 15/01/26, customer group 2": cancel the order line
        - "Product 123456 out of stock, use product 654321": swap products
        - "Available quantity 1000 units": change product quantity
        - "Supercalifragilistico": unknown

        Capture the intention and your confidence on your choice (how confident you are 0.0-1.0)

        """),
        ("human", "Analyze the instruction")
    ])

    try:
        logger.info("MODEL NODE - Invoking model")
        result = (prompt | llm_with_structure).invoke({"instruction": instruction, "customer_group": customer_group})
        intention = result["parsed"].intention
        if intention.lower() == "unknown":
            error_descr = "unknown"
        tool_confidence = result["parsed"].tool_confidence
        total_tokens = result["raw"].usage_metadata["total_tokens"]
    except Exception as e:
        logger.info(f"MODEL NODE - Exception invoking model: {e}")
        intention = "Unknown"
        tool_confidence = 0

    logger.info(f"MODEL NODE - Found intention {intention}")
    logger.info(f"MODEL NODE - Error decription {error_descr}")
    logger.info(f"MODEL NODE - Quitting node")

    return {
        "instruction": instruction,
        "intention": intention,
        "tool_confidence": tool_confidence,
        "messages": [AIMessage(content=f"Intention: {intention}")],
        "total_tokens": total_tokens,
        "error_descr": error_descr,
        "node_name": "MODEL"
    }

def node_choose_tool(state: State) -> dict:
    """
    Node to choose the tool to apply to each instruction.
    """

    # Gets intention from previous node. If no intention identified, return "no tool"
    # Gets the schema of the identified tool
    # Uses the schema to build the prompt sent to the slm
    # if the intention->tool selection is clear, use the tool's schema to request a structured_output
    # Returns the parameters according to the schema

    logger.info(f"CHOOSE TOOL NODE - Starting")

    intention = state["intention"]

    error_descr = ""

    if intention.lower() == "unknown":
        logger.info(f"CHOOSE TOOL NODE - Intention unknown")
        logger.info(f"CHOOSE TOOL NODE - Quitting")
        error_descr = "Unknown intention"
        return {
            "tool_params": None,
            "error_descr": error_descr,
            "messages": AIMessage(content = "Unknown intention"),
            "node_name": "CHOOSE TOOL"
        }

    instruction = state["instruction"]
    tool_confidence = state["tool_confidence"]
    order_id = state["order_id"]
    product_id = state["product_id"]

    schema = TOOL_SCHEMAS[intention]
    schema_dict = schema.model_json_schema()
    schema_json = json.dumps(schema_dict, indent=2)

    llm_with_structure = llm.with_structured_output(schema, include_raw=True)
    # Prompt to capture tool parameters
    prompt = ChatPromptTemplate.from_messages([
        ("system", """
            You are an expert in machine learning. Your task is to extract the parameters of an instruction based on its intention.

            ### INTENTION PARAMETERS DEFINITION:

            {schema_json}

            ### POSSIBLE PARAMETERS TYPES:
            - "order_id": integer number with 9 digits.
            - "product_id": integer number with 6 digits.
            - "date": string with format dd/mm/yyyy.

            ### ADDTIONAL INFORMATION (to complete all parameters use them if needed):
            Order ID: {order_id}
            Product ID: {product_id}

            ### RESTRICTIONS
            If a parameter is not available, return "None" in all cases.


            Instruction: {instruction}
            Intention: {intention}

            Include your reasoning in the answer"""),
        ("human", "Analyze the instruction")
    ])

    try:
        result = (prompt | llm_with_structure).invoke({"schema_json": schema_json,
                                                       "instruction": instruction,
                                                       "intention": intention,
                                                       "order_id": order_id,
                                                       "product_id": product_id})
        tool_parameters = result["parsed"].model_dump()
        tool_parameters.pop("reasoning", None)
        tool_parameters["order_id"] = order_id
        tool_parameters["product_id"] = product_id
        if "None" in tool_parameters.values():
            error_descr = "Unknown parameter"
        total_tokens = result["raw"].usage_metadata["total_tokens"]

    except Exception as e:
        logger.info(f"CHOOSE TOOL NODE - Exception invoking model: {e}")
        print(f"Error: {e}")
        intention = "Unknown"
        parameters = None
        error_descr = "Unknown intention"

    logger.info(f"CHOOSE TOOL NODE - Parameters: {tool_parameters}")
    logger.info(f"CHOOSE TOOL NODE - Quitting")

    return {
        "messages": [AIMessage(content=f"Node choose tool")],
        "tool_params": tool_parameters,
        "params_schema": intention,
        "chosen_tool": state["intention"],
        "total_tokens": total_tokens,
        "error_descr": error_descr,
        "node_name": "HUMAN FEEDBACK"
    }

def node_contact_human(state: State) -> dict:
    """
    Node to build a human readable message requesting for feedback when no tool is selected.
    """

    # Build a human readable message with the conflict
    logger.info(f"HUMAN FEEDBACK NODE - Starting")
    logger.info(f"HUMAN FEEDBACK NODE - Feedback {state["error_descr"]}")
    logger.info(f"HUMAN FEEDBACK NODE - Quitting")

    return {
        "messages": [AIMessage(content=f"Node contact human")],
        "node_name": "HUMAN CONTACT",
        "chosen_tool": None
    }

def tool_node(state: State) -> dict:
    """
    Test graph flow
    """
    logger.info(f"TOOL EXECUTION - Starting")

    tool = tools[state["intention"]]
    result = tool.invoke(state["tool_params"])

    return {
        "messages": [AIMessage(content=f"Test tool_node")],
        "error_descr": result["error_descr"],
        "chosen_tool": result["chosen_tool"],
        "node_name": "TOOL EXECUTION"
    }
#-----------------------------------------------------------------------------------------------------------------------
# Block 6 - Conditional routing
#-----------------------------------------------------------------------------------------------------------------------

def check_intention(state: State) -> Literal["choose_tool", "contact_human"]:
    """
    Checks if intention is clear to avoid calling the model unnecesarily
    """
    logger.info(f"EDGE CHECK CORRECT INTENTION - Starting")
    logger.info(f"EDGE CHECK CORRECT INTENTION - {state["error_descr"]}")
    result = ""
    if len(state["error_descr"])>0:
        result = "contact_human"
    else:
        result = "choose_tool"
    logger.info(f"EDGE CHECK CORRECT INTENTION - Routing to {result}")
    logger.info(f"EDGE CHECK CORRECT INTENTION - Quitting")
    return result

def tool_or_human_calling(state: State) -> Literal["tool_node", "contact_human"]:
    """
    Checks if the model is capable to choose a clear tool with its params. If yes, moves to Tool node.
    If not, move to contact human node.
    """

    # We can go to the tool_node if:
    # 1. There is a clear intention
    # 2. All params are available
    # 3. Confidence is above threshold
    logger.info(f"EDGE CHECK CORRECT TOOL - Starting")
    result = "tool_node"
    if state["intention"].lower() == "unknown":
        result = "contact_human"
    if len(state["error_descr"])>0:
        result = "contact_human"
    if state["tool_confidence"] < CONFIDENCE_THRESHOLD:
        result = "contact_human"

    logger.info(f"EDGE CHECK CORRECT TOOL - Routing to {result}")
    logger.info(f"EDGE CHECK CORRECT TOOL - Quitting")

    return result

def check_correct_execution(state: State) -> Literal["contact_human", "end"]:
    """
    Routes to End if the tool is executed correctly.
    Routes to give feedback to humans in any other case.
    """
    logger.info(f"EDGE CHECK CORRECT EXECUTION- Starting")
    result = ""
    if len(state["error_descr"])>0:
        result = "contact_human"
    else:
        result = "end"
    logger.info(f"EDGE CHECK CORRECT EXECUTION - Routing to {result}")
    logger.info(f"EDGE CHECK CORRECT EXECUTION - Quitting")
    return result


#-----------------------------------------------------------------------------------------------------------------------
# Block 7 - Build the graph
#-----------------------------------------------------------------------------------------------------------------------

builder = StateGraph(State)

# Add nodes
builder.add_node("read_instruction", node_read_instruction)
builder.add_node("choose_tool", node_choose_tool)
builder.add_node("tool_node", tool_node)
builder.add_node("contact_human", node_contact_human)

# Flow definition
builder.add_edge(START, "read_instruction")
builder.add_conditional_edges(
    "read_instruction",
    check_intention,
    {
        "choose_tool": "choose_tool",
        "contact_human": "contact_human"
    }
)
#builder.add_edge("read_instruction", "choose_tool")
builder.add_conditional_edges(
    "choose_tool",
    tool_or_human_calling,
    {
        "tool_node": "tool_node",
        "contact_human": "contact_human"
    }
)
builder.add_conditional_edges(
    "tool_node",
    check_correct_execution,
    {
        "contact_human": "contact_human",
        "end": END
    }
)
builder.add_edge("tool_node", END)
builder.add_edge("contact_human", END)

graph = builder.compile()

#-----------------------------------------------------------------------------------------------------------------------
# Block 8 - Run an Example
#-----------------------------------------------------------------------------------------------------------------------
def print_graph_structure(graph: StateGraph) -> None:
    print("-" * 70)
    print("GRAPH FLOW")
    print("-" * 70)
    print(graph.get_graph().draw_ascii())

def execute_instruction(graph: StateGraph, instruction: dict) -> None:
    """
    Graph execution for one single instruction.
    """
    initial_state = {
        "messages": HumanMessage(content="Processing instruction"),
        "instruction": instruction["instruction"],
        "order_id": instruction["order_id"],
        "product_id": instruction["product_id"],
        "customer_group": instruction["customer_group"],
        "chosen_tool": None,
        "tool_confidence": None,
        "tool_params": None,
        "total_tokens": 0
    }

    timer = time.perf_counter()
    result = graph.invoke(initial_state)
    timer = time.perf_counter() - timer
    hit = str(result["chosen_tool"]) == str(instruction["correct_answer"])

    return {
        "hit": hit,
        "tokens": result["total_tokens"],
        "error_descr": result['error_descr'],
        "tool": result["chosen_tool"],
        "time": timer,
        "chosen_tool": result["chosen_tool"],
    }

instructions = [
    {
        "order_id": 999999999,
        "product_id": 654321,
        "customer_group": 1,
        "instruction": "Check this, please!!!",
        "correct_answer": "None"
    },
    {
        "order_id": 197538624,
        "product_id": 234567,
        "customer_group": 1,
        "instruction": "Product available on 15/01/2026",
        "correct_answer": "delay_order(order_id=197538624, new_date=15/01/2026)"
    },
    {
        "order_id": 842673195,
        "product_id": 234567,
        "customer_group": 2,
        "instruction": "Product available on 15/01/2026",
        "correct_answer": "cancel_order_line(order_id=842673195, product_id=234567)"
    },
    {
        "order_id": 842673195,
        "product_id": 986532,
        "customer_group": 1,
        "instruction": "Product available on 15/01/2026, change to product 123456",
        "correct_answer": "swap_products(order_id=842673195, old_product_id=986532, new_product_id=123456)"
    },
    {
        "order_id": 842673195,
        "product_id": 784512,
        "customer_group": 2,
        "instruction": "Maximum quantity available 3000 units",
        "correct_answer": "change_product_quantity(order_id=842673195, product_id=784512, new_quantity=3000)"
    },
    {
        "order_id": 123456789,
        "product_id": 654321,
        "customer_group": 1,
        "instruction": "Product not available",
        "correct_answer": "None"
    },
    {
        "order_id": 123456789,
        "product_id": 654321,
        "customer_group":2,
        "instruction": "Product not available",
        "correct_answer": "cancel_order_line(order_id=123456789, product_id=654321)"
    },
    {
        "order_id": 987654321,
        "product_id": 123456,
        "customer_group": 1,
        "instruction": "Product not available till 20/01/2026",
        "correct_answer": "delay_order(order_id=987654321, new_date=20/01/2026)"
    },
    {
        "order_id": 987654321,
        "product_id": 123456,
        "customer_group": 1,
        "instruction": "Can't touch this!!!",
        "correct_answer": "None"
    },
    {
        "order_id": 987654321,
        "product_id": 123456,
        "customer_group": 1,
        "instruction": "Release order",
        "correct_answer": "unblock_credit(order_id=987654321)"
    }
]

def single_case_execution(graph: StateGraph)->None:
    """
    UI for one single instruction execution
    """

    def wrapper(order_id, product_id, customer_group, instruction, correct_answer):
        input_dict = {
            "order_id": order_id or "",
            "product_id": product_id or "",
            "customer_group": customer_group or "",
            "instruction": instruction or "",
            "correct_answer": correct_answer or ""
        }

        result = execute_instruction(graph, input_dict)

        with open(LOG_NAME, "r") as f:
            log = f.read()

        logger.info("="*70)    #to separate interactions

        return result["tokens"], f"{result["time"]:.2f}", log, result["chosen_tool"]

    with gr.Blocks(
        theme=gr.themes.Base(),
        css="""
            button { font-size: 16px !important; font-weight: bold !important; }
            .textbox { border: 2px solid #000 !important; }
            label { font-weight: bold !important; }
            .gradio-container footer { display: none !important; }
        """
    )  as TFGApp:
        gr.HTML("""
        <div style='text-align: center; background: #d4d4d8; color: #1f2937;
                    padding: 15px; border-radius: 10px; margin: 20px 0;'>
            <h3 style='font-size: 20px; margin: 0;'>Single System Test</h3>
        </div>
        """)
        with gr.Row(equal_height=True):
            with gr.Column(scale=1, min_width=320):
                gr.Markdown("### Inputs")
                order_id = gr.Textbox(label="Order ID")
                product_id = gr.Textbox(label="Product ID")
                customer_group = gr.Textbox(label="Customer Group")
                instruction = gr.Textbox(label="Instruction")
            with gr.Column(scale=1, min_width=320):
                gr.Markdown("### Outputs")
                tokens = gr.Textbox(label="Tokens")
                time = gr.Textbox(label="Time")
                chosen_tool = gr.Textbox(label="Tool")
        with gr.Row(equal_height=True):
            log = gr.Textbox(label="Log", lines=20)

        run_btn = gr.Button("Run")
        run_btn.click(wrapper,
                  inputs=[order_id, product_id, customer_group, instruction],
                  outputs=[tokens, time, log, chosen_tool]
        )
    TFGApp.launch(share=False, debug=True)

def extended_test(graph: StateGraph, iterations: int, instructions: list) -> None:
    print("*" * 70)
    hits = 0
    tokens = 0
    time_model_invoking = 0
    timer = time.perf_counter()
    for i in range(iterations):
        counter = 0
        for instruction in instructions:
            counter += 1
            data = execute_instruction(graph, instruction)
            hits += data["hit"]
            tokens += data["tokens"]
            time_model_invoking += data["time"]
            print(f"Iteration: {i}.{counter} - Instruction: {instruction["instruction"]} Customer group: {instruction["customer_group"]} Chosen tool: {data["chosen_tool"]} - Correct answer: {instruction["correct_answer"]}")
        random.shuffle(instructions)
    timer = time.perf_counter() - timer

    print("*" * 70)
    print(f"RESULTS AFTER {iterations * len(instructions)} ITERATIONS:")
    total_instructions = iterations * len(instructions)
    print(f"Hits {hits} --> {hits/total_instructions:.1%}")
    print(f"Tokens {tokens} - Average: {tokens/total_instructions:.1f}")
    print(f"Total Model Time: {time_model_invoking}s - Average: {time_model_invoking/total_instructions:.1f}s")
    print(f"Total test time: {timer:.1f}s  Average: {timer/total_instructions:.1f}s ")

#print_graph_structure(graph)
#extended_test(graph, 1, instructions)
single_case_execution(graph)


/tmp/ipython-input-667605517.py:740: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipython-input-667605517.py:740: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.


In [14]:
# Execute when issues connecting the server
import subprocess
import requests
import time

print("🔍 DIAGNÓSTICO: Qwen2.5\n")

# 1. Verificar si Ollama está corriendo
print("1️⃣ Verificando Ollama...")
try:
    response = requests.get('http://localhost:11434/api/tags', timeout=2)
    print("   ✅ Ollama responde")
except:
    print("   ❌ Ollama NO responde - reiniciando...")
    subprocess.Popen(['ollama', 'serve'],
                     stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL)
    time.sleep(5)

# 2. Listar modelos disponibles
print("\n2️⃣ Modelos disponibles:")
result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
print(result.stdout)

if 'qwen2.5' not in result.stdout:
    print("   ❌ Qwen2.5 NO ESTÁ DESCARGADO")
    print("\n3️⃣ Descargando ahora...")
    subprocess.run(['ollama', 'pull', 'qwen2.5:7b'])
else:
    print("   ✅ Qwen2.5 encontrado")

# 3. Verificar vía API
print("\n4️⃣ Verificando vía API...")
response = requests.get('http://localhost:11434/api/tags')
models = response.json().get('models', [])
print(f"   Modelos en API: {[m['name'] for m in models]}")

🔍 DIAGNÓSTICO: Qwen2.5

1️⃣ Verificando Ollama...
   ✅ Ollama responde

2️⃣ Modelos disponibles:
NAME          ID              SIZE      MODIFIED      
qwen2.5:7b    845dbda0ea48    4.7 GB    9 minutes ago    

   ✅ Qwen2.5 encontrado

4️⃣ Verificando vía API...
   Modelos en API: ['qwen2.5:7b']
